In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

import joblib
import os

order_level = pd.read_csv("../data/processed/order_level_dataset.csv")

order_level.head()

order_level.shape

date_columns = [
    "purchase_timestamp",
    "approved_at",
    "delivered_carrier_date",
    "delivered_customer_date",
    "estimated_delivery_date"
]

for col in date_columns:
    order_level[col] = pd.to_datetime(order_level[col], errors="coerce")


order_level["bad_review"] = np.where(order_level["review_score"] <= 2, 1, 0)

order_level["bad_review"].value_counts(normalize=True) * 100


#Late delivery modeling
late_delivery_features = [
    "customer_state",
    "main_product_category",
    "payment_type",
    "product_count",
    "unique_product_count",
    "seller_count",
    "total_price",
    "total_freight",
    "avg_price",
    "total_payment",
    "payment_installments",
    "estimated_delivery_days",
    "purchase_month",
    "purchase_day_of_week",
    "purchase_hour"
]

target = "is_late"

late_df = order_level[late_delivery_features + [target]].copy()

late_df.isnull().sum()

late_df = late_df.dropna()

#Splitting the data into features and target variable
x = late_df[late_delivery_features]
y = late_df[target]

#Checking target balance
y.value_counts(normalize=True) * 100

#Numerical and categorical features
categorical_features = [
    "customer_state",
    "main_product_category",
    "payment_type"
]

numeric_features = [
    "product_count",
    "unique_product_count",
    "seller_count",
    "total_price",
    "total_freight",
    "avg_price",
    "total_payment",
    "payment_installments",
    "estimated_delivery_days",
    "purchase_month",
    "purchase_day_of_week",
    "purchase_hour"
]

#Train-test split
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

#Building preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)


#Training late delivery logistic regression model
late_logistic_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

#Training the logistic regression model
late_logistic_model.fit(x_train, y_train)

#Predicting on the test set
y_pred = late_logistic_model.predict(x_test)
y_proba = late_logistic_model.predict_proba(x_test)[:, 1]

#Evaluating the logistic regression model
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


#Training late delivery random forest model
late_rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])

#Training the random forest model
late_rf_model.fit(x_train, y_train)

#Predicting on the test set
y_pred_rf = late_rf_model.predict(x_test)
y_proba_rf = late_rf_model.predict_proba(x_test)[:, 1]

#Evaluating the random forest model
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf))
print("Recall:", recall_score(y_test, y_pred_rf))
print("F1 Score:", f1_score(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_rf))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))


#Comparing late deliveries models
late_model_results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_test, y_pred),
        accuracy_score(y_test, y_pred_rf)
    ],
    "Precision": [
        precision_score(y_test, y_pred),
        precision_score(y_test, y_pred_rf)
    ],
    "Recall": [
        recall_score(y_test, y_pred),
        recall_score(y_test, y_pred_rf)
    ],
    "F1 Score": [
        f1_score(y_test, y_pred),
        f1_score(y_test, y_pred_rf)
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, y_proba),
        roc_auc_score(y_test, y_proba_rf)
    ]
})

late_model_results

late_model_results.to_csv("../reports/late_delivery_model_results.csv", index=False)


#Bad review modeling
bad_review_features = [
    "customer_state",
    "main_product_category",
    "payment_type",
    "product_count",
    "unique_product_count",
    "seller_count",
    "total_price",
    "total_freight",
    "avg_price",
    "total_payment",
    "payment_installments",
    "delivery_days",
    "estimated_delivery_days",
    "delay_days",
    "is_late",
    "purchase_month",
    "purchase_day_of_week",
    "purchase_hour"
]

target = "bad_review"

review_df = order_level[bad_review_features + [target]].copy()
review_df = review_df.dropna()

#Splitting the data into features and target variable
x = review_df[bad_review_features]
y = review_df[target]

#Checking target balance
y.value_counts(normalize=True) * 100

#Categorical and numerical features
categorical_features_review = [
    "customer_state",
    "main_product_category",
    "payment_type"
]

numeric_features_review = [
    "product_count",
    "unique_product_count",
    "seller_count",
    "total_price",
    "total_freight",
    "avg_price",
    "total_payment",
    "payment_installments",
    "delivery_days",
    "estimated_delivery_days",
    "delay_days",
    "is_late",
    "purchase_month",
    "purchase_day_of_week",
    "purchase_hour"
]

#Train-test split
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

#Building preprocessor
review_preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features_review),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features_review)
    ]
)

#Training bad review logistic regression model
bad_review_logistic_model = Pipeline(steps=[
    ("preprocessor", review_preprocessor),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

#Training the logistic regression model
bad_review_logistic_model.fit(x_train, y_train)

#Predicting on the test set
y_pred_review = bad_review_logistic_model.predict(x_test)
y_proba_review = bad_review_logistic_model.predict_proba(x_test)[:, 1]

#Evaluating the logistic regression model
print("Accuracy:", accuracy_score(y_test, y_pred_review))
print("Precision:", precision_score(y_test, y_pred_review))
print("Recall:", recall_score(y_test, y_pred_review))
print("F1 Score:", f1_score(y_test, y_pred_review))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_review))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_review))


#Training bad review random forest model
bad_review_rf_model = Pipeline(steps=[
    ("preprocessor", review_preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])

#Training the random forest model
bad_review_rf_model.fit(x_train, y_train)

#Predicting on the test set
y_pred_review_rf = bad_review_rf_model.predict(x_test)
y_proba_review_rf = bad_review_rf_model.predict_proba(x_test)[:, 1]

#Evaluating the random forest model
print("Accuracy:", accuracy_score(y_test, y_pred_review_rf))
print("Precision:", precision_score(y_test, y_pred_review_rf))
print("Recall:", recall_score(y_test, y_pred_review_rf))
print("F1 Score:", f1_score(y_test, y_pred_review_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_review_rf))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_review_rf))


#Comparing bad review models
bad_review_model_results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_review),
        accuracy_score(y_test, y_pred_review_rf)
    ],
    "Precision": [
        precision_score(y_test, y_pred_review),
        precision_score(y_test, y_pred_review_rf)
    ],
    "Recall": [
        recall_score(y_test, y_pred_review),
        recall_score(y_test, y_pred_review_rf)
    ],
    "F1 Score": [
        f1_score(y_test, y_pred_review),
        f1_score(y_test, y_pred_review_rf)
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, y_proba_review),
        roc_auc_score(y_test, y_proba_review_rf)
    ]
})

bad_review_model_results

bad_review_model_results.to_csv("../reports/bad_review_model_results.csv", index=False)


#Customer segmentation using RFM
snapshot_date = order_level["purchase_timestamp"].max() + pd.Timedelta(days=1)

rfm = order_level.groupby("customer_unique_id").agg(
    recency=("purchase_timestamp", lambda x: (snapshot_date - x.max()).days),
    frequency=("order_id", "nunique"),
    monetary=("total_payment", "sum")
).reset_index()

rfm.head()

#Scaling RFM features
rfm_features = ["recency", "frequency", "monetary"]

scaler = StandardScaler()

rfm_scaled = scaler.fit_transform(rfm[rfm_features])

#Finding best number of clusters
inertia = []

K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(rfm_scaled)
    inertia.append(kmeans.inertia_)

#Elbow curve
plt.figure(figsize=(8, 5))
plt.plot(K_range, inertia, marker="o")
plt.title("Elbow Method for Customer Segmentation")
plt.xlabel("Number of Clusters")
plt.ylabel("Inertia")
plt.tight_layout()
plt.show()


#Training KMeans with 4 clusters
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)

rfm["cluster"]=kmeans.fit_predict(rfm_scaled)

rfm["cluster"].value_counts()

#Analyzing cluster characteristics
cluster_summary = rfm.groupby("cluster").agg(
    customer_count=("customer_unique_id", "count"),
    avg_recency=("recency", "mean"),
    avg_frequency=("frequency", "mean"),
    avg_monetary=("monetary", "mean")
).reset_index()

cluster_summary

#Labeling customer segments based on RFM characteristics
def label_customer_segment(row):
    if row["frequency"] >= 2 and row["monetary"] >= rfm["monetary"].quantile(0.75):
        return "High Value Customer"
    elif row["recency"] > rfm["recency"].quantile(0.75):
        return "At Risk Customer"
    elif row["frequency"] >= 2:
        return "Repeat Customer"
    else:
        return "One-Time Customer"
    
rfm["customer_segment"] = rfm.apply(label_customer_segment, axis=1)

rfm["customer_segment"].value_counts()

rfm.to_csv("../data/processed/customer_segments.csv", index=False)
cluster_summary.to_csv("../reports/customer_segmentation_summary.csv", index=False)


#Visualizing segments
segment_summary = (
    rfm.groupby("customer_segment")
    .agg(
        customer_count=("customer_unique_id", "count"),
        avg_recency=("recency", "mean"),
        avg_frequency=("frequency", "mean"),
        avg_monetary=("monetary", "mean")
    )
    .reset_index()
)

segment_summary

plt.figure(figsize=(10, 5))
sns.barplot(data=segment_summary, x="customer_segment", y="customer_count")
plt.xticks(rotation=30)
plt.title("Customer Segment Distribution")
plt.xlabel("Customer Segment")
plt.ylabel("Number of Customers")
plt.tight_layout()
plt.show()

segment_summary.to_csv("../reports/customer_segment_distribution.csv", index=False)


#Saving ML models
os.makedirs("../models", exist_ok=True)

joblib.dump(late_rf_model, "../models/late_delivery_model.pkl")
joblib.dump(bad_review_rf_model, "../models/bad_review_model.pkl")
joblib.dump(kmeans, "../models/customer_segmentation_kmeans.pkl")
joblib.dump(scaler, "../models/rfm_scaler.pkl")


#ML summary report
ml_summary = pd.DataFrame({
    "Model": [
        "Late Delivery Prediction",
        "Bad Review Prediction",
        "Customer Segmentation"
    ],
    "Problem Type": [
        "Classification",
        "Classification",
        "Clustering"
    ],
    "Target / Output": [
        "is_late",
        "bad_review",
        "customer_segment"
    ],
    "Business Use Case": [
        "Predict orders likely to be delivered late",
        "Predict orders likely to receive poor customer review",
        "Group customers by purchase behavior"
    ]
})

ml_summary

ml_summary.to_csv("../reports/phase_5_ml_summary.csv", index=False)



Accuracy: 0.6545558204623199
Precision: 0.11563845050215207
Recall: 0.6166794185156848
F1 Score: 0.1947565543071161
ROC-AUC: 0.6925228536855808

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.66      0.78     17987
           1       0.12      0.62      0.19      1307

    accuracy                           0.65     19294
   macro avg       0.54      0.64      0.49     19294
weighted avg       0.90      0.65      0.74     19294



# Phase 5 Conclusion

In Phase 5, machine learning models were developed for logistics risk prediction, customer satisfaction prediction, and customer segmentation.

Models created:

1. Late Delivery Prediction
   - Target: is_late
   - Algorithms: Logistic Regression, Random Forest
   - Business use: Identify orders likely to be delivered late.

2. Bad Review Prediction
   - Target: bad_review
   - Algorithms: Logistic Regression, Random Forest
   - Business use: Identify orders at risk of poor customer satisfaction.

3. Customer Segmentation
   - Method: RFM + KMeans
   - Business use: Segment customers into groups for retention and marketing.

The trained models were saved for future use in the Streamlit application.